In [7]:
# 두 키워드 파일의 기사 ID가 모두 묶음 파일에 들어 있는지 확인
import glob, unicodedata, pandas as pd

DATA_DIR = "."   # 제공 CSV가 있는 폴더로 바꾸기
ID_COL = "원문URL"

def ids(path):
    s, empty = set(), 0
    for c in pd.read_csv(path, encoding="utf-8-sig", usecols=[ID_COL], chunksize=50_000):
        col = c[ID_COL].dropna().astype(str).str.strip()
        empty += len(c) - len(col)
        s.update(col)
    print(f"{path[:40]}… 행 {len(s)+empty:,} 중 URL 빈 행 {empty:,}")
    return s

b, k1, k2 = ids(bundle), ids(kw1), ids(kw2)
print("묶음:", len(b), "| 키워드 합집합:", len(k1 | k2))
print("묶음에 없는 키워드 기사:", len((k1 | k2) - b))

./02_AI반도체_묶음_AI반도체_120057거… 행 120,053 중 URL 빈 행 0
./02_AI반도체_키워드_인공지능반도체… 행 51,194 중 URL 빈 행 0
./02_AI반도체_키워드_AI반도체_109193거… 행 109,189 중 URL 빈 행 0
묶음: 120053 | 키워드 합집합: 120053
묶음에 없는 키워드 기사: 0


In [ ]:
# 후보 22건을 묶음 파일에서 찾아 본문 속 '저작권' 횟수 세기 (이 셀 하나로 끝)
import glob, re, unicodedata
import pandas as pd

# ① 묶음 파일 찾기
bundle = next(p for p in glob.glob("../data/provided/*.csv") if "묶음_AI반도체" in unicodedata.normalize("NFC", p))

# ② 후보 22건 목록 읽기
short = pd.read_csv("../report/news_shortlist.csv", encoding="utf-8-sig")
urls = set(short["원문URL"])

# ③ 묶음 파일을 나눠 읽으며 22건만 골라내기
parts = []
for c in pd.read_csv(bundle, encoding="utf-8-sig", usecols=["원문URL", "본문전문"], chunksize=20_000):
    parts.append(c[c["원문URL"].isin(urls)])
full = pd.concat(parts).drop_duplicates("원문URL")

# ④ '저작권' 횟수 세기 (방송 고지문 "저작권은 ○○에 있습니다"는 빼고)
boiler = re.compile(r"저작권은[^.]{0,20}있습니다")
full["저작권_횟수"] = full["본문전문"].fillna("").map(lambda t: boiler.sub("", t).count("저작권"))
full["본문길이"] = full["본문전문"].str.len()

# ⑤ 결과 표 출력
result = short.merge(full[["원문URL", "저작권_횟수", "본문길이"]], on="원문URL")
print(result[["tier", "row", "저작권_횟수", "본문길이", "제목"]].to_string())

   tier  row  저작권_횟수   본문길이                                                제목
0     A  304       3   2031              “누구나 AI 학습에 이용 가능”…공공저작물 활용 범위 대폭 확대
1     A  542       3   2542                경총 AI 저작권 규제 완화 필요…로봇 원본데이터 활용 허용도
2     A  422       5   3284                 국가대표 독파모 AI 증류 '불법' 소지…美와 선제 협상해야
3     A  197       7   1418               AI 학습용 뉴스도 정식 구매…언론진흥재단, ‘뉴스토어’ 알린다
4     A   12       5   1353                   AI복제물 저작권 면책 놓고…IT·출판업계 논쟁 '후끈'
5     A  456       8   4446      [AI 한쿡] 정부는 소버린 AI 승부수... 업계는 '저작권·규제'와 줄다리기
6     A  239       1   2467      '소라'로 미키·아이언맨 만든다… 오픈AI-디즈니 '동맹'[글로벌 모닝 브리핑]
7     A  285       0   2027              AI가 '지브리풍 뚝딱' 시대에…한국 AI 스타트업은 다죽어 왜?
8     A  165       1   1231        어도비, 표절 안하는 AI 이미지 프로그램 출시… 구글도 챗봇 ‘바드’ 출격
9     A  511       2   3247                    챗GPT, 유튜브 100만 시간 봤어? [Global]
10    A  435      10   3528  [IS인터뷰] 이성엽 AI윤리법제포럼 회장, 선·악 없는 AI, 인간 창의력 넘볼 수도
11    A  159       2   1134                           - AI저작권 등 

In [2]:
# 묶음 CSV의 열 이름(첫 덩어리만) + 두 PDF의 쪽수와 샘플 쪽 글자를 출력해 pypdf 추출 품질 확인
import unicodedata, pandas as pd
from pathlib import Path
from pypdf import PdfReader
bundle = next(p for p in Path("../data/provided").iterdir() if unicodedata.normalize("NFC", p.name).startswith("02_AI반도체"))
first = next(pd.read_csv(bundle, encoding="utf-8-sig", chunksize=5))
print("묶음 열:", first.columns.tolist()); print(first.iloc[0].astype(str).str[:80].to_string(), "\n")
for f, page in [("gov_genai_copyright_guide_2023.pdf", 10), ("gov_fair_use_guide_2026.pdf", 7)]:
    r = PdfReader(f"../data/gov/{f}")
    print(f"=== {f}: {len(r.pages)}쪽, {page+1}번째 쪽 앞 500자 ===")
    print(r.pages[page].extract_text()[:500], "\n")

묶음 열: ['제목', '본문전문', '원문URL', '발행일', '매체명', '문서종류', '수집키워드']
제목                               국산 AI 칩·보드, 서울대 교육 과정에 활용
본문전문     서울대가 국산 인공지능(AI) 반도체 칩과 보드를 고급 AI 반도체 인재 양성에 활...
원문URL    https://n.news.naver.com/mnews/article/030/000...
발행일                                    2023-12-19T07:23:00
매체명                                                   전자신문
문서종류                                                  news
수집키워드                                     AI 반도체; 인공지능 반도체 

=== gov_genai_copyright_guide_2023.pdf: 109쪽, 11번째 쪽 앞 500자 ===
11
생성형 AI 저작권 안내서
제5장Q&A 제6장부 록 제4장 제3장 제2장 제1장
생성형 AI 
기술과 저작권
제1장
  
인간의 생각이나 감정이 표현된 창작물, 즉 저작물에 대한 권리로서 창작자에게 귀속하는 것이 원칙이며, 저작
인격권(공표권, 성명표시권, 동일성유지권)과 저작재산권(복제권, 공연권, 공중송신권, 전시권, 배포권, 대여권, 
2차적저작물작성권)으로 구성
저작권
나   AI 산출물의 저작권 침해 쟁점
   생성형 AI 산출물은 이용자의 프롬프트 등 입력에 따라서 학습 완료된 AI 모델로부터 확률적으로 
도출된 것
   -  AI 산출물이 학습용 데이터에 포함된 원저작물의 일부와 같거나 유사한 경우 원저작물에 대한 저작권 
침해에 해당하는지, 해당할 경우 침해자는 누구인지 등과 관련한 저작권 쟁점 발생
  AI를 통해서 만들어 낸 결과물을 지칭하는 다양한 용어들이 사용되고 있으나, 우리 저작권법

In [3]:
# gov_01·gov_02에서 50자 미만인 쪽 번호와 남은 글자(앞 40자)를 출력
import sqlite3, pandas as pd
with sqlite3.connect("../store/docs.db") as con:
    p = pd.read_sql("SELECT doc_id, page, n_chars, text FROM gov_pages WHERE n_chars < 50", con)
for doc, g in p.groupby("doc_id"):
    print(f"=== {doc}: {g.page.tolist()}")
    for r in g.itertuples():
        print(f"  {r.page:>3}쪽 ({r.n_chars}자): {r.text.replace(chr(10), ' / ')[:40]}")

=== gov_01: [1, 6, 7, 13, 14, 15, 22, 23, 27, 28, 29, 38, 39, 46, 47, 52, 53, 69, 70, 71, 72, 73, 76, 77, 80, 81, 86, 87, 99, 100, 101, 104, 105, 109]
    1쪽 (30자): 생성형 AI / 저작권 / 안내서 / 저작권관계자료 2023-10
    6쪽 (14자): 생성형 AI / 저작권 / 안내서
    7쪽 (18자): 생성형 AI / 기술과 저작권 / 제1장
   13쪽 (33자): 생성형 AI / 기술과 저작권 / 제1장 / 생성형 AI / 저작권 / 
   14쪽 (14자): 생성형 AI / 저작권 / 안내서
   15쪽 (19자): AI 사업자에 대한 / 안내사항 / 제2장
   22쪽 (14자): 생성형 AI / 저작권 / 안내서
   23쪽 (17자): 저작권자에 대한 / 안내사항 / 제3장
   27쪽 (32자): 저작권자에 대한 / 안내사항 / 제3장 / 생성형 AI / 저작권 / 안
   28쪽 (14자): 생성형 AI / 저작권 / 안내서
   29쪽 (19자): AI 이용자에 대한 / 안내사항 / 제4장
   38쪽 (14자): 생성형 AI / 저작권 / 안내서
   39쪽 (18자): AI 산출물과 / 저작권 등록 / 제5장
   46쪽 (14자): 생성형 AI / 저작권 / 안내서
   47쪽 (39자): 「AI-저작권 제도개선 워킹그룹」 / 운영 경과 및 향후 추진 계획 / 
   52쪽 (14자): 생성형 AI / 저작권 / 안내서
   53쪽 (3자): Q&A
   69쪽 (14자): 생성형 AI / 저작권 / 안내서
   70쪽 (14자): 생성형 AI / 저작권 / 안내서
   71쪽 (2자): 부록
   72쪽 (14자): 생성형 AI / 저작권 / 안내서
   73쪽 (49자): 인공 신경망 / (Artificial Neural Network)을 / 
   76쪽 (14자): 생성형 AI / 저작권 / 

In [ ]:
# 절 제목·Q&A 질문의 시작 모양 확인: gov_01 41쪽(제5장 02), 66쪽(Q&A 12), gov_02 8쪽의 줄 앞부분 출력

import sqlite3
with sqlite3.connect("../store/docs.db") as con:
    for doc, page in [("gov_01", 41), ("gov_01", 66), ("gov_02", 9)]:
        t = con.execute("SELECT text FROM gov_pages WHERE doc_id=? AND page=?", (doc, page)).fetchone()[0]
        print(f"=== {doc} {page}쪽 ===")
        for line in t.splitlines()[:15]:
            print("  |", line[:50])

# pypdf가 글을 페이지 안에서 위-아래 순서대로 뽑지 못한 경우가 있음
# [쪽 단위 (짧은 쪽은 다음 쪽과 합침), 장 이름은 표지 지도로 붙임] 으로 정부 안내서 pdf 청킹

=== gov_01 41쪽 ===
  | AI 산출물과
  | 저작권 등록
  | 제5장
  | 어떠한 표현 행위에도 인간의 창작적 기여가 있었다고 볼 수 없는 AI 산출물에 대한 저작권
  | - 저작권 등록은 인간의 사상 또는 감정이 표현된 창작물에 대해서만 가능
  | ※ 저작권은 저작물의 창작과 동시에 발생하는 것으로 등록·납본·기탁 등 일체의 절차나 방식
  | (저작권법 제10조 제2항, 무방식주의), 저작권 등록이 되었다고 하여 저작물이 아닌 것에
  | 내지 저작권 부여의 효과가 발생하는 것도 아님
  | AI 산출물의 저작권 등록 가능 여부
  | 저작물에 관한 일정한 사항(저작자 성명, 창작연월일, 맨 처음 공표연월일 등)과 저작재산권
  | 질권설정 등 권리의 변동에 대한 사항을 저작권등록부라는 공적인 장부에 등재하고 일반 국민들
  | 열람할 수 있도록 공시하는 것을 의미
  | 등록을 할 경우 저작자로 등록된 자는 저작자로 추정되는 효과(추정력), 권리 변동 사실 등
  | 당사자가 아닌 제3자에게도 주장할 수 있게 되는 효과(대항력), 소송상 원고가 실손해를 증
  | 법에서 정한 금액을 손해액으로 인정할 수 있도록 하는 효과(법정손해배상제도) 등 발생
=== gov_01 66쪽 ===
  | AI 산출물은 저작물로서 인정되지 않기 때문에 저작권 등록을 할 수 없습니다.
  | 저작권법에서는 인간이 만든 창작성 있는 저작물만을 저작권 등록의 대상으로 하고 있기
  | 때문에 AI가 만들어낸 산출물은 저작권 등록을 할 수 없습니다.
  | 한국저작권위원회는 AI 산출물에 대해 등록 신청한 사안에서 이를 반려한 바 있으며 미국
  | 저작권청에서도 AI 산출물을 저작권 등록 신청한 건에 대해 등록을 거절한 바 있습니다.
  | 제5장 03. AI 산출물의 저작권 등록 관련 국내외 사례 부분 참고
  | Q12 AI 산출물을 저작권 등록할 수 있나요?
=== gov_02 9쪽 ===
  | 2. 생성형
  | 인공지능 학습의
  | 개념과 

In [ ]:
# 뉴스 16건 본문의 마지막 100자를 출력해 고지문·기자 정보가 남아 있는지 확인

import sqlite3, pandas as pd
with sqlite3.connect("../store/docs.db") as con:
    n = pd.read_sql("SELECT doc_id, source, text FROM news_provided", con)
for r in n.itertuples():
    print(f"{r.doc_id} ({r.source}): ...{' '.join(r.text.split())[-100:]}")


# 기사를 더 보고 싶다면? 기사 섹션 분류 안내 이 기사를 추천합니다"(네이버 화면 문구) 정보가 같이 딸려옴
# chunk.py에서 필터 추가

news_01 (헤럴드경제): ...응이 필요하다"며 "과학기술관계장관회의를 중심으로 부처 간 역량을 결집해 우리나라가 보유한 AI 반도체 제조 경쟁력을 하나로 모으고, 정책을 신속하게 추진해 나가겠다"고 강조했다.
news_02 (뉴스1): ...업들의 이익을 대변하고 노사 관계를 조율하기 위해 설립된 대표적인 종합 경제단체로 경영계의 입장을 취합해 정부에 정책을 건의하거나 노동 고용 규제 관련 법안에 목소리를 내고 있다.
news_03 (뉴스1): ... 금융 의료 에너지 공공 행정 등 분야는 해외 클라우드와 모델에 데이터를 맡기기 어려운 만큼 자국 법 문화 보안 규제를 반영한 자체 AI 체계가 필요하다는 논리가 힘을 얻고 있다.
news_04 (ZDNet Korea): ...관 약 350개사가 참여해 500여개 부스를 운영하며 AI 코어, 에이전틱 AI, 피지컬 AI, AI 보안, 국방 AI, AI 반도체 등 다양한 기술과 산업 적용 사례를 선보인다.
news_05 (매일경제): ...지 않는 범위에서 AI가 저작물을 자유롭게 이용하게 하되, 가 사후에 이를 거부할 수 있도록 해야 한다"고 절충안을 제안하기도 했다. 기사를 더 보고 싶다면? 기사 섹션 분류 안내
news_06 (디지털데일리): ...능이 50% 이상 뛰어난 차세대 항염증 단백질 후보물질을 개발했습니다. 동물실험 결과 뇌 신경염증 회복에도 탁월한 효과를 보여 난치성 염증 질환 치료제로서의 가능성을 입증했습니다.
news_07 (서울경제): ... "미국은 세계 기술 공급망 전체에 걸쳐 안전하고 회복력 있으며 혁신이 주도하는 생태계를 구축한다는 원칙을 중심으로 국가 간 연합(coalition)을 조직하고 있다"고 강조했다.
news_08 (문화일보): ... 개발 업체로 잘 알려진 엔비디아는 이제 AI의 두뇌 역할로 쓰이는 GPU 시장의 90% 이상을 점유하고 있다. 기사를 더 보고 싶다면? 기사 섹션 분류 안내 이 기사를 추천합니다
news_09 (더스쿠프): ...밝혔다. 하지만 지금은 

In [6]:
# eval_set.jsonl의 정부 문서 문항에 gold_pages({문서: [쪽]})를 추가 (gold_section 기준, 사람 확인 후)

import json
R = lambda a, b: list(range(a, b + 1))
PAGES = {"G01": {"gov_01": [41, 66]}, "G02": {"gov_01": R(31, 32)}, "G03": {"gov_02": R(26, 28)},
         "G04": {"gov_02": [3]}, "S01": {"gov_02": [8, 44]}, "S02": {"gov_01": R(41, 44)},
         "S03": {"gov_02": R(14, 17)}, "S04": {"gov_01": R(29, 37)},
         "S05": {"gov_01": [32], "gov_02": [54]}, "S06": {"gov_02": [59]}}
p = "../eval/eval_set.jsonl"
rows = [json.loads(l) for l in open(p, encoding="utf-8")]
for r in rows:
    if r["id"] in PAGES:
        r["gold_pages"] = PAGES[r["id"]]
with open(p, "w", encoding="utf-8") as f:
    f.writelines(json.dumps(r, ensure_ascii=False) + "\n" for r in rows)
print(sum("gold_pages" in r for r in rows), "문항에 gold_pages 추가")

10 문항에 gold_pages 추가
